<div style="text-align:center;">
  <h1 size=10>
    <b>BIG DATA ANALYTICS PROJECT</b><br>
    <b>Money Laundring Detection - IBM Transactions</b>
  </h1>
</div>

<h2 style="text-align:center;">
Master's in Data Science and Advanced Analytics - NOVA IMS (25/26)
</h2>

**Group 26**
- Bárbara Franco (20250388)
- Catarina Mendinhas (20250422)
- Maria Miguel Fonseca (20250380)
- Rodrigo Santos (20250387)
- Rodrigo Teixeira (20250393)

**GitHub repository:** https://github.com/mariamiguel720/Big-Data-Analytics-Project-25-26

<font color='#2f94d7' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>

- [1. Project Overview](#1)
- [2. Set Up & Import Libraries](#2)
- [3. Load Data](#3)
- [4. Data Exploration](#4)
- [5. Data Preprocessing](#5)


# <font color='#2f94d7' size=6>**1. Project Overview**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

This project focuses on money laundring detecting and uses the dataset IBM Transactions for Anti Money Laundering (HI-Small-Trans.csv) which contains approximately 5 million transactions with 11 features. GraphFrames and Streaming tecnhniques will be employed.

# <font color='#2f94d7' size=6>**2. Setup & Import Libraries**</font> <a class="anchor" id="2"></a>

[Back to TOC](#toc)

In [1]:
!pip install "pyspark==3.5.0" 

In [2]:
# INSTALL JAVA 17

!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless
!java -version

Hit:1 https://packages.cloud.google.com/apt cloud-sdk InRelease
Hit:2 https://download.docker.com/linux/ubuntu noble InRelease      
Get:3 https://cli.github.com/packages stable InRelease [3917 B]                
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Hit:5 https://archive.ubuntu.com/ubuntu noble InRelease                        
Hit:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease  
Hit:7 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://security.ubuntu.com/ubuntu noble-security InRelease              
Hit:9 https://archive.ubuntu.com/ubuntu noble-updates InRelease                
Hit:10 http://deb.wakemeops.com/wakemeops stable InRelease          
Hit:11 https://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:12 https://cloud.archive.ubuntu.com/ubuntu noble InRelease
Hit:13 https://cloud.archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:14 https://cloud.archive.ubuntu.c

In [3]:
%pip install graphframes-py==0.10.0

Note: you may need to restart the kernel to use updated packages.


In [4]:
# IMPORT LIBRARIES
import os

# import the package we just installed
from graphframes import *

# import data types - All data types of Spark SQL are located in the package of pyspark.sql.types
from pyspark.sql.types import *

# Row can be used to create a row object by using named arguments
from pyspark.sql import Row

from pyspark.sql.functions import col

from pyspark.sql import SparkSession

from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [5]:
# SET JAVA_HOME AND INITIALIZE SPARK SESSION
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

spark = (
    SparkSession.builder
    .master("local[*]")          
    .appName("GraphFrames-AML")
    .config("spark.jars.packages", "io.graphframes:graphframes-spark3_2.12:0.10.0")
    .config("spark.driver.memory", "6g")                               
    .config("spark.sql.shuffle.partitions", "8")                                       
    .config("spark.sql.files.maxPartitionBytes", "128m")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN") 
print(f"Spark version: {spark.version}")

:: loading settings :: url = jar:file:/system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/zeus/.ivy2/cache
The jars for the packages stored in: /home/zeus/.ivy2/jars
io.graphframes#graphframes-spark3_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-41451211-692c-4c2e-a688-214e91280750;1.0
	confs: [default]
	found io.graphframes#graphframes-spark3_2.12;0.10.0 in central
	found io.graphframes#graphframes-graphx-spark3_2.12;0.10.0 in central
:: resolution report :: resolve 360ms :: artifacts dl 23ms
	:: modules in use:
	io.graphframes#graphframes-graphx-spark3_2.12;0.10.0 from central in [default]
	io.graphframes#graphframes-spark3_2.12;0.10.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   || 

Spark version: 3.5.0


In [6]:
sc = spark.sparkContext

# GraphFrames connected components requires a checkpoint directory
sc.setCheckpointDir("/tmp/graphframes-checkpoints")

spark.conf.set("spark.sql.shuffle.partitions", "16")

# <font color='#2f94d7' size=6>**3. Load Data**</font> <a class="anchor" id="3"></a>

[Back to TOC](#toc)

In [7]:
DATA_DIR = "../../data/IBM_Transactions/HI-Small_Trans.csv"

df = spark.read.csv(DATA_DIR, header = True, inferSchema = True, multiLine = True, escape = '"')

print("Loaded file.")
print(f"Data size: ({df.count()}, {len(df.columns)})")

Loaded file.


Data size: (5078345, 11)


The dataset has approximately 5 million records and 11 features

# <font color='#2f94d7' size=6>**4. Data Exploration**</font> <a class="anchor" id="4"></a>

[Back to TOC](#toc)

In [8]:
df.printSchema()

root
 |-- Timestamp: string (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account2: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account4: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)



Timestamp has to be converted to the timestamp, the other columns seem to be in the correct format. nullable = True means that the columns can accept null values

In [9]:
df.show(10)

26/05/24 16:47:17 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Timestamp, From Bank, Account, To Bank, Account, Amount Received, Receiving Currency, Amount Paid, Payment Currency, Payment Format, Is Laundering
 Schema: Timestamp, From Bank, Account2, To Bank, Account4, Amount Received, Receiving Currency, Amount Paid, Payment Currency, Payment Format, Is Laundering
Expected: Account2 but found: Account
CSV file: file:///teamspace/studios/this_studio/Big-Data-Analytics-Project-25-26/data/IBM_Transactions/HI-Small_Trans.csv


+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+-------------+
|       Timestamp|From Bank| Account2|To Bank| Account4|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+-------------+
|2022/09/01 00:20|       10|8000EBD30|     10|8000EBD30|        3697.34|         US Dollar|    3697.34|       US Dollar|  Reinvestment|            0|
|2022/09/01 00:20|     3208|8000F4580|      1|8000F5340|           0.01|         US Dollar|       0.01|       US Dollar|        Cheque|            0|
|2022/09/01 00:00|     3209|8000F4670|   3209|8000F4670|       14675.57|         US Dollar|   14675.57|       US Dollar|  Reinvestment|            0|
|2022/09/01 00:02|       12|8000F5030|     12|8000F5030|        2806.97|         US Dollar|    2806.

Spark is giving a warning about having 2 columns with the same name, "Account". Spark renames it to "Account2", "Account4". Let's rename the columns for a more clean and understandable way.

In [10]:
# RENAME COLUMNS
df = df.withColumnRenamed("Account2", "src_account") \
       .withColumnRenamed("Account4", "dst_account") \
       .withColumnRenamed("From Bank", "from_bank") \
       .withColumnRenamed("To Bank", "to_bank") \
       .withColumnRenamed("Amount Received", "amount_received") \
       .withColumnRenamed("Amount Paid", "amount_paid") \
       .withColumnRenamed("Receiving Currency", "recv_currency") \
       .withColumnRenamed("Payment Currency", "pay_currency") \
       .withColumnRenamed("Payment Format", "pay_format") \
       .withColumnRenamed("Is Laundering", "is_laundering")

In [37]:
print("Time Period")
print("=" * 50)
df.select(
    F.min("Timestamp").alias("start"),
    F.max("Timestamp").alias("end")
).show()

Time Period


+----------------+----------------+
|           start|             end|
+----------------+----------------+
|2022/09/01 00:00|2022/09/18 16:18|
+----------------+----------------+



The transactions go from 01/09/2022 to 18/09/2022, consisting of only 18 days.

# <font color='#2f94d7' size=5>**4.1. Missing Values**</font> <a class="anchor" id="4_1"></a>

[Back to TOC](#toc)

In [11]:
# CHECK FOR NULL VALUES
print("NULL VALUES PER COLUMN")
print("=" * 50)
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

NULL VALUES PER COLUMN


26/05/24 16:47:20 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Timestamp, From Bank, Account, To Bank, Account, Amount Received, Receiving Currency, Amount Paid, Payment Currency, Payment Format, Is Laundering
 Schema: Timestamp, From Bank, Account2, To Bank, Account4, Amount Received, Receiving Currency, Amount Paid, Payment Currency, Payment Format, Is Laundering
Expected: Account2 but found: Account
CSV file: file:///teamspace/studios/this_studio/Big-Data-Analytics-Project-25-26/data/IBM_Transactions/HI-Small_Trans.csv


+---------+---------+-----------+-------+-----------+---------------+-------------+-----------+------------+----------+-------------+
|Timestamp|from_bank|src_account|to_bank|dst_account|amount_received|recv_currency|amount_paid|pay_currency|pay_format|is_laundering|
+---------+---------+-----------+-------+-----------+---------------+-------------+-----------+------------+----------+-------------+
|        0|        0|          0|      0|          0|              0|            0|          0|           0|         0|            0|
+---------+---------+-----------+-------+-----------+---------------+-------------+-----------+------------+----------+-------------+



# <font color='#2f94d7' size=5>**4.2. Duplicates**</font> <a class="anchor" id="4_2"></a>

[Back to TOC](#toc)

In [12]:
print("DUPLICATES")
print("=" * 50)
total = df.count()
distinct_rows = df.dropDuplicates().count()
print(f"Total nr of rows    : {total:,}")
print(f"Nr of unique rows   : {distinct_rows:,}")
print(f"Nr of duplicates    : {total - distinct_rows:,}")

DUPLICATES


26/05/24 16:47:43 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Timestamp, From Bank, Account, To Bank, Account, Amount Received, Receiving Currency, Amount Paid, Payment Currency, Payment Format, Is Laundering
 Schema: Timestamp, From Bank, Account2, To Bank, Account4, Amount Received, Receiving Currency, Amount Paid, Payment Currency, Payment Format, Is Laundering
Expected: Account2 but found: Account
CSV file: file:///teamspace/studios/this_studio/Big-Data-Analytics-Project-25-26/data/IBM_Transactions/HI-Small_Trans.csv


Total nr of rows    : 5,078,345
Nr of unique rows   : 5,078,336
Nr of duplicates    : 9


In [13]:
# Encontrar as linhas que aparecem mais do que uma vez
duplicates = df.groupBy(df.columns).count().filter(F.col("count") > 1)
# groups all rows that have the same values in every column and counts how many times it happens, then filters for the groups 
# more than 1 distinct row

print(f"Duplicated groups: {duplicates.count()}")
duplicates.show(truncate=False)

26/05/24 16:48:39 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Timestamp, From Bank, Account, To Bank, Account, Amount Received, Receiving Currency, Amount Paid, Payment Currency, Payment Format, Is Laundering
 Schema: Timestamp, From Bank, Account2, To Bank, Account4, Amount Received, Receiving Currency, Amount Paid, Payment Currency, Payment Format, Is Laundering
Expected: Account2 but found: Account
CSV file: file:///teamspace/studios/this_studio/Big-Data-Analytics-Project-25-26/data/IBM_Transactions/HI-Small_Trans.csv


Duplicated groups: 9


26/05/24 16:49:20 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Timestamp, From Bank, Account, To Bank, Account, Amount Received, Receiving Currency, Amount Paid, Payment Currency, Payment Format, Is Laundering
 Schema: Timestamp, From Bank, Account2, To Bank, Account4, Amount Received, Receiving Currency, Amount Paid, Payment Currency, Payment Format, Is Laundering
Expected: Account2 but found: Account
CSV file: file:///teamspace/studios/this_studio/Big-Data-Analytics-Project-25-26/data/IBM_Transactions/HI-Small_Trans.csv


+----------------+---------+-----------+-------+-----------+---------------+-------------+-----------+------------+----------+-------------+-----+
|Timestamp       |from_bank|src_account|to_bank|dst_account|amount_received|recv_currency|amount_paid|pay_currency|pay_format|is_laundering|count|
+----------------+---------+-----------+-------+-----------+---------------+-------------+-----------+------------+----------+-------------+-----+
|2022/09/07 21:25|29992    |8099A29B1  |220    |813725AE1  |3.0E-6         |Bitcoin      |3.0E-6     |Bitcoin     |Bitcoin   |0            |2    |
|2022/09/01 16:20|12004    |800C927C1  |12004  |800C927C0  |8.0E-6         |Bitcoin      |0.08       |Euro        |ACH       |0            |2    |
|2022/09/01 16:20|12004    |800C927C1  |220    |813D8C1E1  |8.0E-6         |Bitcoin      |8.0E-6     |Bitcoin     |Bitcoin   |0            |2    |
|2022/09/09 10:03|6075     |80C702911  |6075   |80C702910  |2.0E-6         |Bitcoin      |0.02       |US Dollar   |ACH

9 duplicates were found. The number is very small when compared to the size of the dataset, the best practice is too remove these 9 rows. 

# <font color='#2f94d7' size=5>**4.3. Is Laundring Class Imbalance**</font> <a class="anchor" id="4_3"></a>

[Back to TOC](#toc)

In [ ]:
# CHECK FOR IS LAUNDERING CLASS IMBALANCE
print("DISTRIBUTION OF IS LAUNDERING")
print("=" * 50)
df.groupBy("is_laundering").count() \
  .withColumn("percentage", F.round(F.col("count") / total * 100, 2)) \
  .orderBy("is_laundering").show()

DISTRIBUTION OF IS LAUNDERING


+-------------+-------+----------+
|is_laundering|  count|percentage|
+-------------+-------+----------+
|            0|5073168|      99.9|
|            1|   5177|       0.1|
+-------------+-------+----------+



Only 0.1% of the transactions is money laundering, this is an expected class imbalance for this type of data.

# <font color='#2f94d7' size=5>**4.4. Categorical Variables**</font> <a class="anchor" id="4_4"></a>

[Back to TOC](#toc)

In [21]:
# CHECK FOR THE DIFFERENT CURRENCIES
df.groupBy("pay_currency") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(truncate=False)

+-----------------+-------+
|pay_currency     |count  |
+-----------------+-------+
|US Dollar        |1895172|
|Euro             |1168297|
|Swiss Franc      |234860 |
|Yuan             |213752 |
|Shekel           |192184 |
|Rupee            |190202 |
|UK Pound         |180738 |
|Yen              |155209 |
|Ruble            |155178 |
|Bitcoin          |146066 |
|Canadian Dollar  |140042 |
|Australian Dollar|136769 |
|Mexican Peso     |110159 |
|Saudi Riyal      |89014  |
|Brazil Real      |70703  |
+-----------------+-------+



In [23]:
# CHECK FOR THE DIFFERENT CURRENCIES
df.groupBy("recv_currency") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(truncate=False)

+-----------------+-------+
|recv_currency    |count  |
+-----------------+-------+
|US Dollar        |1879341|
|Euro             |1172017|
|Swiss Franc      |237884 |
|Yuan             |206551 |
|Shekel           |194988 |
|Rupee            |192065 |
|UK Pound         |181255 |
|Ruble            |157361 |
|Yen              |156319 |
|Bitcoin          |148151 |
|Canadian Dollar  |141357 |
|Australian Dollar|138511 |
|Mexican Peso     |111030 |
|Saudi Riyal      |89971  |
|Brazil Real      |71544  |
+-----------------+-------+



The receiving and payment currencies have same values but with different frequencies. For both the US dollar is the most used.

In [ ]:
# CHECK THE TRANSACTIONS WITH DIFFERENT PAYMENT AND RECEIVING CURRENCIES
# Number of transactions with different payment and receiving currencies
cross_currency = df.filter(col("pay_currency") != col("recv_currency")).count()

print(f"Number of transactions with different payment and receiving currencies: {cross_currency}")
print(f"Percentage of transactions with different payment and receiving currencies: {cross_currency/total * 100}%")

Number of transactions with different payment and receiving currencies: 72170
Percentage of transactions with different payment and receiving currencies: 1.4211322783308342%


In [28]:
# CHECK FOR MOST COMMON CURRENCY PAIRS
print("MOST COMMON CURRENCY PAIRS")
print("="*50)
df.groupBy("pay_currency","recv_currency").count().orderBy("count",ascending=False).show(30, truncate=False)

MOST COMMON CURRENCY PAIRS


+-----------------+-----------------+-------+
|pay_currency     |recv_currency    |count  |
+-----------------+-----------------+-------+
|US Dollar        |US Dollar        |1856392|
|Euro             |Euro             |1153708|
|Swiss Franc      |Swiss Franc      |234429 |
|Yuan             |Yuan             |203522 |
|Shekel           |Shekel           |192066 |
|Rupee            |Rupee            |189006 |
|UK Pound         |UK Pound         |177939 |
|Ruble            |Ruble            |154852 |
|Yen              |Yen              |153603 |
|Bitcoin          |Bitcoin          |146013 |
|Canadian Dollar  |Canadian Dollar  |139065 |
|Australian Dollar|Australian Dollar|136478 |
|Mexican Peso     |Mexican Peso     |109656 |
|Saudi Riyal      |Saudi Riyal      |88891  |
|Brazil Real      |Brazil Real      |70555  |
|US Dollar        |Euro             |15838  |
|Euro             |US Dollar        |11060  |
|Yuan             |US Dollar        |6675   |
|US Dollar        |Yuan           

In [ ]:
# ANALYSE THE DIFFERENT CURRENCY PAIRS
print("MOST COMMON DIFFERENT CURRENCY PAIRS")
print("="*50)
df.filter(col("pay_currency") != col("recv_currency")) \
    .groupBy("pay_currency","recv_currency") \
    .count() \
    .orderBy("count",ascending=False) \
    .show(30, truncate=False)

MOST COMMON CURRENCY PAIRS


+---------------+-----------------+-----+
|pay_currency   |recv_currency    |count|
+---------------+-----------------+-----+
|US Dollar      |Euro             |15838|
|Euro           |US Dollar        |11060|
|Yuan           |US Dollar        |6675 |
|US Dollar      |Yuan             |2547 |
|US Dollar      |Swiss Franc      |2507 |
|US Dollar      |UK Pound         |2489 |
|US Dollar      |Rupee            |2310 |
|US Dollar      |Shekel           |2204 |
|US Dollar      |Yen              |2027 |
|US Dollar      |Ruble            |1934 |
|UK Pound       |US Dollar        |1772 |
|US Dollar      |Canadian Dollar  |1531 |
|US Dollar      |Bitcoin          |1496 |
|US Dollar      |Australian Dollar|1391 |
|Yuan           |Euro             |1230 |
|Yen            |US Dollar        |1045 |
|US Dollar      |Mexican Peso     |911  |
|US Dollar      |Saudi Riyal      |838  |
|Rupee          |US Dollar        |803  |
|US Dollar      |Brazil Real      |757  |
|Canadian Dollar|US Dollar        

Most transactions are done from US Dollar to US Dollar and Euro to Euro, followed by the Swiss Franc.

The most common pairs of different payment and receiving currencies are by far the ones with US Dollar and Euro, followed by Yuan and US Dollar.

In [24]:
# ANALYSE THE DIFFERENT PAYMENT FORMATS
print("DIFFERENT PAYMENT FORMATS")
print("=" * 50)
df.groupBy("pay_format") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(truncate=False)

DIFFERENT PAYMENT FORMATS


+------------+-------+
|pay_format  |count  |
+------------+-------+
|Cheque      |1864331|
|Credit Card |1323324|
|ACH         |600797 |
|Cash        |490891 |
|Reinvestment|481056 |
|Wire        |171855 |
|Bitcoin     |146091 |
+------------+-------+



In [31]:
# ANALYSE RELATIONSHIP BETWEEN PAYMENT CURRENCY, RECEIVING CURRENCY AND PAYMENT FORMAT
df.groupBy("pay_currency","recv_currency","pay_format").count().orderBy("count", ascending = False).show()

+------------+-------------+------------+------+
|pay_currency|recv_currency|  pay_format| count|
+------------+-------------+------------+------+
|   US Dollar|    US Dollar|      Cheque|708957|
|   US Dollar|    US Dollar| Credit Card|506535|
|        Euro|         Euro|      Cheque|441177|
|        Euro|         Euro| Credit Card|315559|
|   US Dollar|    US Dollar|         ACH|201645|
|   US Dollar|    US Dollar|        Cash|187884|
|   US Dollar|    US Dollar|Reinvestment|186391|
|     Bitcoin|      Bitcoin|     Bitcoin|146013|
|        Euro|         Euro|         ACH|126434|
|        Euro|         Euro|Reinvestment|116658|
|        Euro|         Euro|        Cash|114005|
| Swiss Franc|  Swiss Franc|      Cheque| 89702|
|        Yuan|         Yuan|      Cheque| 78278|
|      Shekel|       Shekel|      Cheque| 74436|
|       Rupee|        Rupee|      Cheque| 73537|
|    UK Pound|     UK Pound|      Cheque| 68596|
|   US Dollar|    US Dollar|        Wire| 64957|
| Swiss Franc|  Swis

The most common combination is payment and receiving currency in US Dollars with cheque as the payment format. In this table, only results with the same payment and receiving currencies are shown (they are not on the top 20).

In [34]:
# ANALYSE RELATIONSHIP BETWEEN PAYMENT CURRENCY, RECEIVING CURRENCY AND PAYMENT FORMAT
df.filter(col("pay_currency") != col("recv_currency")) \
    .groupBy("pay_currency","recv_currency","pay_format") \
    .count() \
    .orderBy("count", ascending = False) \
    .show()

+------------+-----------------+----------+-----+
|pay_currency|    recv_currency|pay_format|count|
+------------+-----------------+----------+-----+
|   US Dollar|             Euro|       ACH|15838|
|        Euro|        US Dollar|       ACH|11060|
|        Yuan|        US Dollar|       ACH| 6675|
|   US Dollar|             Yuan|       ACH| 2547|
|   US Dollar|      Swiss Franc|       ACH| 2507|
|   US Dollar|         UK Pound|       ACH| 2489|
|   US Dollar|            Rupee|       ACH| 2310|
|   US Dollar|           Shekel|       ACH| 2204|
|   US Dollar|              Yen|       ACH| 2027|
|   US Dollar|            Ruble|       ACH| 1934|
|    UK Pound|        US Dollar|       ACH| 1772|
|   US Dollar|  Canadian Dollar|       ACH| 1531|
|   US Dollar|Australian Dollar|       ACH| 1391|
|        Yuan|             Euro|       ACH| 1230|
|   US Dollar|          Bitcoin|       ACH| 1099|
|         Yen|        US Dollar|       ACH| 1045|
|   US Dollar|     Mexican Peso|       ACH|  911|


The most common transactions with different payment and receiving currencies are all done by ACH (eletronic transfering, that does the transfering in batches instead of sending all the money at once, popular in the USA).

In [ ]:
# CHECK RELATIONSHIP BETWEEN IS LAUNDERING AND PAYMENT FORMAT
print("LAUNDERING TRANSACTIONS BY FORMAT")
print("=" * 50)
df.filter(F.col("is_laundering") == 1) \
  .groupBy("pay_format").count() \
  .orderBy(F.desc("count")).show()

ACH is the payment format most associated with money laundering, with it's number of transactions being almost 14 times bigger than the second most common format. And ACH is not the most popular payment format as we have seen, its Cheque and Credit Card.

In [35]:
print("BANKS WITH THE MOST TRANSACTIONS")
print("=" * 50)
df.groupBy("from_bank").count() \
  .orderBy(F.desc("count")).show(10)

BANKS WITH THE MOST TRANSACTIONS


+---------+------+
|from_bank| count|
+---------+------+
|       70|449859|
|       10| 81629|
|       12| 79754|
|        1| 62211|
|       15| 52511|
|      220| 52417|
|       20| 41008|
|        3| 38413|
|        7| 31086|
|      211| 30451|
+---------+------+
only showing top 10 rows



The bank identified with the number 70 is by far the most common one, with its total number of transactions being almost 10% of the total number of transactions in the data, and the bank in second place has only approximately 1.5% of the total transactions.

In [41]:
# CHECK MOST ACTIVE ACOUNTS (PAYMENT)
print("MOST ACTIVE ACCOUNTS(PAYMENT)")
print("=" * 50)

df.groupBy("src_account").count() \
  .orderBy(F.desc("count")).show(10)

MOST ACTIVE ACCOUNTS(PAYMENT)


26/05/24 18:16:38 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Account
 Schema: Account2
Expected: Account2 but found: Account
CSV file: file:///teamspace/studios/this_studio/Big-Data-Analytics-Project-25-26/data/IBM_Transactions/HI-Small_Trans.csv


+-----------+------+
|src_account| count|
+-----------+------+
|  100428660|168672|
|  1004286A8|103018|
|  100428978| 20497|
|  1004286F0| 18663|
|  100428780| 17264|
|  1004289C0| 16794|
|  100428810| 16426|
|  1004287C8| 14174|
|  100428738| 13756|
|  100428A51| 13073|
+-----------+------+
only showing top 10 rows



The first two accounts have performed more than 100,000 transactions in 18 days ! This is a very suspicious behavior.

In [42]:
# CHECK MOST ACTIVE ACOUNTS (RECEIVING)
print("MOST ACTIVE ACCOUNTS(RECEIVING)")
print("=" * 50)

df.groupBy("dst_account").count() \
  .orderBy(F.desc("count")).show(10)

MOST ACTIVE ACCOUNTS(RECEIVING)


26/05/24 18:16:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Account
 Schema: Account4
Expected: Account4 but found: Account
CSV file: file:///teamspace/studios/this_studio/Big-Data-Analytics-Project-25-26/data/IBM_Transactions/HI-Small_Trans.csv


+-----------+-----+
|dst_account|count|
+-----------+-----+
|  100428660| 1084|
|  1004286A8|  653|
|  80F47A310|  159|
|  100428978|  150|
|  8018859B0|  144|
|  1004289C0|  132|
|  100428780|  117|
|  100428810|  114|
|  80F0EF460|  109|
|  1004286F0|  108|
+-----------+-----+
only showing top 10 rows



The values do not go even close to the ones from the payment table, these are much smaller.

# <font color='#2f94d7' size=5>**4.5. Numeric Variables**</font> <a class="anchor" id="4_5"></a>

[Back to TOC](#toc)

# <font color='#2f94d7' size=6>**5. Data Preprocessing**</font> <a class="anchor" id="5"></a>

[Back to TOC](#toc)

In [ ]:
# CONVERT COLUMN "Timestamp" to Timestamp TYPE
df = df.withColumn("Timestamp", F.to_timestamp("Timestamp", "yyyy/MM/dd HH:mm"))

df = df.dropDuplicates()
print(f"Linhas após remover duplicados: {df.count():,}")